# COOTEFOO — Visual Exploration


In [5]:
from pathlib import Path

import pandas as pd
import altair as alt
import json
  # <-- modifica qui se i file .json sono in un'altra cartella

alt.data_transformers.disable_max_rows()
alt.renderers.enable("default")


RendererRegistry.enable('default')

In [6]:
persons = pd.read_json("../public/data/persons.json")
organizations = pd.read_json("../public/data/organizations.json")
places = pd.read_json("../public/data/places.json")
topics = pd.read_json("../public/data/topics.json")
meetings = pd.read_json("../public/data/meetings.json")
trips = pd.read_json("../public/data/trips.json")
trip_stops = pd.read_json("../public/data/trip_stops.json")
initiatives = pd.read_json("../public/data/initiatives.json")
timeline = pd.read_json("../public/data/initiative_status_timeline.json")
initiative_participants = pd.read_json("../public/data/initiative_participants.json")
initiatives = pd.read_json("../public/data/initiatives.json")


for name, df in [("persons", persons), ("organizations", organizations), ("places", places),
                  ("topics", topics), ("meetings", meetings), ("trips", trips),
                  ("trip_stops", trip_stops), ("initiatives", initiatives),
                  ("timeline", timeline), ("initiative_participants", initiative_participants)]:
    print(f"{name:26s} {len(df):5d} righe")


persons                        7 righe
organizations                 10 righe
places                       173 righe
topics                        15 righe
meetings                      16 righe
trips                        342 righe
trip_stops                  1363 righe
initiatives                   79 righe
timeline                     101 righe
initiative_participants      113 righe


## 1. Persone mancanti nei dataset rispetto a journalist


In [7]:
person_coverage = persons.melt(
    id_vars=["id", "role"], value_vars=["in_filah", "in_trout"],
    var_name="dataset", value_name="presente"
)
person_coverage["dataset"] = person_coverage["dataset"].str.replace("in_", "").str.upper()

chart = alt.Chart(person_coverage).mark_rect(stroke="white", strokeWidth=1).encode(
    x=alt.X("dataset:N", title=None),
    y=alt.Y("id:N", title="Persona", sort=persons["id"].tolist()),
    color=alt.Color("presente:N", title="Presente",
                     scale=alt.Scale(domain=[True, False], range=["#2ca02c", "#d62728"])),
    tooltip=["id", "role", "dataset", "presente"],
).properties(title="Copertura membri COOTEFOO per dataset", width=180, height=220)
chart


alt.Chart(...)

**Lettura**: FILAH copre solo metà del board (Seal, Simone Kat, Carol Limpet), TROUT li
copre tutti e 6 — la stessa asimmetria di campionamento trovata all'inizio, ora visibile
in una riga di codice invece che ricostruita dal grafo.


## 2. Copertura dei meeting per dataset


In [8]:
meeting_coverage = meetings.melt(
    id_vars=["id", "date_label"], value_vars=["in_filah", "in_trout"],
    var_name="dataset", value_name="presente"
)
meeting_coverage["dataset"] = meeting_coverage["dataset"].str.replace("in_", "").str.upper()
meeting_coverage["n"] = meeting_coverage["id"].str.extract(r"(\d+)").astype(int)

chart = alt.Chart(meeting_coverage).mark_rect(stroke="white", strokeWidth=1).encode(
    x=alt.X("dataset:N", title=None),
    y=alt.Y("id:N", title="Meeting", sort=alt.SortField("n")),
    color=alt.Color("presente:N", title="Presente",
                     scale=alt.Scale(domain=[True, False], range=["#2ca02c", "#d62728"])),
    tooltip=["id", "dataset", "presente"],
).properties(title="Copertura dei meeting per dataset", width=180, height=320)
chart


alt.Chart(...)

**Lettura**: FILAH si ferma al Meeting 12 (mancano gli ultimi 4); TROUT ha un buco nel
mezzo (13-15) ma include il 16 — pattern già trovato, qui a colpo d'occhio.


## 3. Iniziative per industria, per dataset


In [9]:
ip_ind = initiative_participants.explode("industry").dropna(subset=["industry"])

# un'iniziativa e' "nota" per una industry in un dataset se ALMENO UN partecipante
# di quella iniziativa/industry ha in_filah (o in_trout) = True
agg = (
    ip_ind.groupby(["initiative_id", "industry"])
    .agg(in_filah=("in_filah", "any"), in_trout=("in_trout", "any"))
    .reset_index()
)

industry_coverage = pd.DataFrame({
    "journalist": agg.groupby("industry")["initiative_id"].nunique(),
    "FILAH": agg[agg.in_filah].groupby("industry")["initiative_id"].nunique(),
    "TROUT": agg[agg.in_trout].groupby("industry")["initiative_id"].nunique(),
}).fillna(0).astype(int).reset_index()

industry_long = industry_coverage.melt(id_vars="industry", var_name="dataset", value_name="n_iniziative")

chart = alt.Chart(industry_long).mark_bar().encode(
    x=alt.X("dataset:N", title=None),
    y=alt.Y("n_iniziative:Q", title="# iniziative distinte"),
    color=alt.Color("dataset:N", title="Dataset",
                     scale=alt.Scale(domain=["journalist", "FILAH", "TROUT"],
                                      range=["#7f7f7f", "#1f77b4", "#ff7f0e"])),
    column=alt.Column("industry:N", title=None),
    tooltip=["industry", "dataset", "n_iniziative"],
).properties(width=80, height=220, title="Iniziative per industria, per dataset")
chart


alt.Chart(...)

TROUT ha meno iniziative sul turismo. TROUT infatti sostiene che cotefoo sia biased verso fishing


considerando small + large vessel, FILAH e TROUT hanno lo stesso numero di iniziative (24)

## 4. Sentiment per persona per industria, per dataset

Heatmap del sentiment medio (persona × industria), affiancata per i tre dataset — il test
diretto di bias: un membro con sentiment sistematicamente positivo su un'industria e
negativo sull'altra mostra esattamente il pattern che TROUT/FILAH si accusano a vicenda.


In [10]:
def sentiment_table_for(dataset_flag_col):
    if dataset_flag_col is None:
        sub = ip_ind
        label = "journalist"
    else:
        sub = ip_ind[ip_ind[dataset_flag_col]]
        label = "FILAH" if dataset_flag_col == "in_filah" else "TROUT"
    t = (sub.dropna(subset=["sentiment"])
            .groupby(["entity_id", "entity_type", "industry"])["sentiment"].mean()
            .reset_index())
    t["dataset"] = label
    return t


sentiment_long = pd.concat([
    sentiment_table_for(None),
    sentiment_table_for("in_filah"),
    sentiment_table_for("in_trout"),
], ignore_index=True)

# etichetta leggibile per il facet, invece del type tecnico "entity.person"/"entity.organization"
sentiment_long["entity_type_label"] = sentiment_long["entity_type"].map({
    "entity.person": "Persone", "entity.organization": "Organizzazioni",
})

chart = alt.Chart(sentiment_long).mark_rect().encode(
    x=alt.X("industry:N", title=None),
    y=alt.Y("entity_id:N", title=None),
    color=alt.Color("sentiment:Q", title="Sentiment medio",
                     scale=alt.Scale(scheme="redyellowgreen", domain=[-1, 1])),
    tooltip=["entity_id", "industry", "dataset", alt.Tooltip("sentiment:Q", format=".2f")],
    row=alt.Row("entity_type_label:N", title=None,
                sort=["Persone", "Organizzazioni"]),
    column=alt.Column("dataset:N", title=None, sort=["journalist", "FILAH", "TROUT"]),
).properties(width=140, height=120).resolve_scale(y="independent")
chart

alt.Chart(...)

I tre silenziati da TROUT (Carol Limpet, Tante Titan, Simone kat) inclinano verso il turismo. Chi TROUT lascia parlare: Ed Helpsford (verde scuro ovunque, anche sul turismo) e Teddy Goldstein (verde su fishing ma arancione/negativo sul turismo). TROUT infatti sostiene che cotefoo sia biased verso fishing

Invece FILAH rimuove Ed Helpsford (molto verde su fishing), tante titan (verde su fishing) e teddy goldstein (verde su fishing e rosso su turismo). FILAH infatti sostiene che cotefoo sia biased verso tourism

In [11]:

# unisco i partecipanti al topic della loro iniziativa (via initiatives.topic_id),
# poi al nome leggibile del topic (via topics.short_topic)
ip_topic = (
    initiative_participants
    .merge(initiatives[["id", "topic_id"]], left_on="initiative_id", right_on="id", suffixes=("", "_init"))
    .merge(topics[["id", "short_topic"]], left_on="topic_id", right_on="id", suffixes=("", "_topic"))
)


def sentiment_table_for(dataset_flag_col):
    if dataset_flag_col is None:
        sub = ip_topic
        label = "journalist"
    else:
        sub = ip_topic[ip_topic[dataset_flag_col]]
        label = "FILAH" if dataset_flag_col == "in_filah" else "TROUT"
    t = (sub.dropna(subset=["sentiment"])
            .groupby(["entity_id", "short_topic"])["sentiment"].mean()
            .reset_index())
    t["dataset"] = label
    return t


sentiment_topic_long = pd.concat([
    sentiment_table_for(None),
    sentiment_table_for("in_filah"),
    sentiment_table_for("in_trout"),
], ignore_index=True)

chart = alt.Chart(sentiment_topic_long).mark_rect().encode(
    x=alt.X("short_topic:N", title=None, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("entity_id:N", title=None),
    color=alt.Color("sentiment:Q", title="Sentiment medio",
                     scale=alt.Scale(scheme="redyellowgreen", domain=[-1, 1])),
    tooltip=["entity_id", "short_topic", "dataset", alt.Tooltip("sentiment:Q", format=".2f")],
    column=alt.Column("dataset:N", title=None, sort=["journalist", "FILAH", "TROUT"]),
).properties(width=280, height=260)
chart

alt.Chart(...)

A livello di topic si vede dove esattamente Simone Kat cambia segno: rosso scuro (-1) su affordable_housing, arancione su new_crane_lomark  (temi di uso del suolo/infrastruttura, non pesca in sé) mentre è verde acceso su tutto ciò che è turistico (heritage_walking_tour, marine_life_deck, seafood_festival, waterfront_market). La sua "opposizione alla pesca" è in realtà più specificamente opposizione a progetti infrastrutturali/abitativi che competono con lo spazio urbano

## 5. Viaggi: dove va ogni membro, ed è riportato nei due dataset?

Prima, quanti viaggi per persona sono noti a ciascun dataset:


In [12]:
trip_coverage = trips.melt(
    id_vars=["id", "person_id"], value_vars=["in_filah", "in_trout"],
    var_name="dataset", value_name="presente"
)
trip_coverage["dataset"] = trip_coverage["dataset"].str.replace("in_", "").str.upper()

trip_counts = (
    trip_coverage[trip_coverage["presente"]]
    .groupby(["person_id", "dataset"]).size().reset_index(name="n_trip")
)
# aggiungo il totale journalist come riferimento
totale = trips.groupby("person_id").size().reset_index(name="n_trip")
totale["dataset"] = "journalist"
trip_counts_all = pd.concat([trip_counts, totale], ignore_index=True)

chart = alt.Chart(trip_counts_all).mark_bar().encode(
    x=alt.X("dataset:N", title=None,
            sort=["journalist", "FILAH", "TROUT"]),
    y=alt.Y("n_trip:Q", title="# viaggi"),
    color=alt.Color("dataset:N",
                     scale=alt.Scale(domain=["journalist", "FILAH", "TROUT"],
                                      range=["#7f7f7f", "#1f77b4", "#ff7f0e"])),
    column=alt.Column("person_id:N", title=None),
    tooltip=["person_id", "dataset", "n_trip"],
).properties(width=60, height=220, title="Numero di viaggi per persona, per dataset")
chart


alt.Chart(...)

In [14]:
with open("../public/data/raw_data/oceanus_map.geojson") as f:
    oceanus_geo = json.load(f)

stops_full = (
    trip_stops
    .merge(trips[["id", "person_id"]], left_on="trip_id", right_on="id", suffixes=("", "_trip"))
    .merge(places[["id", "name", "latitude", "longitude", "zone"]], left_on="place_id", right_on="id",
           suffixes=("", "_place"))
)

background = alt.Chart(alt.Data(values=oceanus_geo["features"])).mark_geoshape(
    fill="#e8e8e8", stroke="white", strokeWidth=1
).project(type="identity", reflectY=True)

person_dropdown = alt.binding_select(options=sorted(persons["id"].unique()), name="Persona: ")
person_select = alt.selection_point(fields=["person_id"], bind=person_dropdown,
                                     value=[{"person_id": persons["id"].iloc[0]}])

points = alt.Chart(stops_full).mark_circle(size=90, opacity=0.8).encode(
    longitude="longitude:Q", latitude="latitude:Q",
    color=alt.Color("zone:N", title="Zona"),
    tooltip=["name", "zone", "time"],
).add_params(person_select).transform_filter(person_select).project(type="identity", reflectY=True)

chart = (background + points).properties(width=500, height=450,
                                          title="Tappe dei viaggi sulla mappa di Oceanus")
chart

alt.LayerChart(...)

In [15]:
zone_counts = stops_full.groupby(["person_id", "zone"]).size().reset_index(name="n_tappe")

chart = alt.Chart(zone_counts).mark_bar().encode(
    x=alt.X("person_id:N", title=None),
    y=alt.Y("n_tappe:Q", stack="normalize", title="Quota di tappe per zona"),
    color=alt.Color("zone:N", title="Zona"),
    tooltip=["person_id", "zone", "n_tappe"],
).properties(width=400, height=300, title="In che zone viaggia ciascun membro (quota %)")
chart

alt.Chart(...)

In [16]:
# escludo le tappe in zona 'government' (probabile effetto "riunione obbligatoria del board",
# non una scelta discrezionale del singolo membro)
stops_no_gov = stops_full[stops_full["zone"] != "government"]

zone_counts_no_gov = stops_no_gov.groupby(["person_id", "zone"]).size().reset_index(name="n_tappe")

chart = alt.Chart(zone_counts_no_gov).mark_bar().encode(
    x=alt.X("person_id:N", title=None),
    y=alt.Y("n_tappe:Q", stack="normalize", title="Quota di tappe per zona"),
    color=alt.Color("zone:N", title="Zona"),
    tooltip=["person_id", "zone", "n_tappe"],
).properties(width=400, height=300,
             title="In che zone viaggia ciascun membro, esclusa 'government' (quota %)")
chart

alt.Chart(...)

rimuovendo government vediamo che Tante Titan è quello piu incline alle zone turistiche--->infatti era assente in TROUT

Anche Teddy Goldstein viaggia in zone turstiche, anche se ha un'opinione negativa su questo

Simone Kat è quella che viaggia di piu in zone industriali, anche se ha opinione negativa su large vessel

Carol Limpet viaggia in zone commerciali. Ha opinione positiva su small vessel

# VERIFICA BIAS

considerando che una persona può ripetere la stessa opinione su un topic piu volte

In [17]:
ip_ind["macro_industry"] = ip_ind["industry"].map({
    "small vessel": "fishing", "large vessel": "fishing", "tourism": "tourism",
})
ip_ind = ip_ind.dropna(subset=["sentiment"])


def sums_for(flag_col):
    sub = ip_ind if flag_col is None else ip_ind[ip_ind[flag_col]]
    label = "journalist" if flag_col is None else ("FILAH" if flag_col == "in_filah" else "TROUT")
    s = sub.groupby("macro_industry")["sentiment"].sum().reset_index()
    s["dataset"] = label
    return s


"""sums_all = pd.concat([sums_for(None), sums_for("in_filah"), sums_for("in_trout")], ignore_index=True)

chart = alt.Chart(sums_all).mark_bar().encode(
    x=alt.X("dataset:N", title=None, sort=["journalist", "FILAH", "TROUT"]),
    y=alt.Y("sentiment:Q", title="Somma dei sentiment"),
    color=alt.Color("macro_industry:N", title="Categoria",
                     scale=alt.Scale(domain=["fishing", "tourism"], range=["#1f77b4", "#2ca02c"])),
    xOffset="macro_industry:N",
    tooltip=["dataset", "macro_industry", alt.Tooltip("sentiment:Q", format=".1f")],
).properties(width=300, height=300, title="Somma sentiment: fishing vs tourism, per dataset")
chart"""

'sums_all = pd.concat([sums_for(None), sums_for("in_filah"), sums_for("in_trout")], ignore_index=True)\n\nchart = alt.Chart(sums_all).mark_bar().encode(\n    x=alt.X("dataset:N", title=None, sort=["journalist", "FILAH", "TROUT"]),\n    y=alt.Y("sentiment:Q", title="Somma dei sentiment"),\n    color=alt.Color("macro_industry:N", title="Categoria",\n                     scale=alt.Scale(domain=["fishing", "tourism"], range=["#1f77b4", "#2ca02c"])),\n    xOffset="macro_industry:N",\n    tooltip=["dataset", "macro_industry", alt.Tooltip("sentiment:Q", format=".1f")],\n).properties(width=300, height=300, title="Somma sentiment: fishing vs tourism, per dataset")\nchart'

In [18]:
def person_topic_sentiment(ip_df, init_df):
    """evita di contare piu' volte lo stesso sentiment quando una persona si esprime
    su piu' iniziative distinte dello stesso topic"""
    merged = ip_df.merge(
        init_df[["id", "topic_id"]], left_on="initiative_id", right_on="id", suffixes=("", "_i")
    )
    return merged.groupby(["entity_id", "entity_type", "topic_id"]).agg(
        sentiment=("sentiment", "first"),
        reason=("reason", "first"),
        industry=("industry", "first"),
        n_initiatives=("initiative_id", "nunique"),
        in_filah=("in_filah", "any"),
        in_trout=("in_trout", "any"),
    ).reset_index()


opinione su un topic ripetuta una volta sola

In [19]:
pts = person_topic_sentiment(initiative_participants, initiatives)

# esplodo industry SOLO dopo la deduplicazione per topic
ip_ind = pts.explode("industry").dropna(subset=["industry"])
ip_ind["macro_industry"] = ip_ind["industry"].map({
    "small vessel": "fishing", "large vessel": "fishing", "tourism": "tourism",
})
ip_ind = ip_ind.dropna(subset=["sentiment"])


def sums_for(flag_col):
    sub = ip_ind if flag_col is None else ip_ind[ip_ind[flag_col]]
    label = "journalist" if flag_col is None else ("FILAH" if flag_col == "in_filah" else "TROUT")
    s = sub.groupby("macro_industry")["sentiment"].sum().reset_index()
    s["dataset"] = label
    return s


sums_all = pd.concat([sums_for(None), sums_for("in_filah"), sums_for("in_trout")], ignore_index=True)

chart = alt.Chart(sums_all).mark_bar().encode(
    x=alt.X("dataset:N", title=None, sort=["journalist", "FILAH", "TROUT"]),
    y=alt.Y("sentiment:Q", title="Somma dei sentiment"),
    color=alt.Color("macro_industry:N", title="Categoria",
                     scale=alt.Scale(domain=["fishing", "tourism"], range=["#1f77b4", "#2ca02c"])),
    xOffset="macro_industry:N",
    tooltip=["dataset", "macro_industry", alt.Tooltip("sentiment:Q", format=".1f")],
).properties(width=300, height=300, title="Somma sentiment: fishing vs tourism, per dataset (deduplicato per topic)")
chart

alt.Chart(...)